In [1]:
from langchain_openai import ChatOpenAI  
from langchain_core.messages import HumanMessage

LLM = ChatOpenAI(model="gpt-4o-mini")   # Initialize the chat model

LLM.invoke([HumanMessage(content="안녕, 잘 지냈어?")]) # Send a message to the model

AIMessage(content='안녕하세요! 저는 잘 지내고 있습니다. 당신은 어떻게 지내세요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 15, 'total_tokens': 32, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_36b9eb89d1', 'id': 'chatcmpl-EdUkgjFICQQJKQg2WZ9csIyIGmpKR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17bff-9585-70b1-94fa-6cc935d5c739-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 17, 'total_tokens': 32, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool       # @tool decorator uses langchain's tool system
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각 반환 함수
    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 LLM 답변 생성에 사용됨
    """
    Time_zone = pytz.timezone(timezone)
    now = datetime.now(Time_zone).strftime("%Y-%m-%d %H:%M:%S")

    # Return format: 'Asia/Seoul (Seoul) 현재시각 2024-06-01 15:30:00'
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} '
     
    print(location_and_local_time)
    return location_and_local_time

In [3]:
# Append the tool to a list and a dictionary
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# Bind the tools to the LLM 
LLM_with_tools = LLM.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# Create messages for the conversation
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야."),
    HumanMessage("인천은 지금 몇시야?"),
]

# Call the LLM_with_tools to generate a response
response = LLM_with_tools.invoke(messages)
messages.append(response)

# Print the conversation messages
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야.', additional_kwargs={}, response_metadata={}), HumanMessage(content='인천은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 138, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ccaf3fe85e', 'id': 'chatcmpl-EdUmejFXoaRyYZW1ufXY0O2Ok3m6C', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a17c01-6fae-7663-8ed6-72b1d9507b04-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezo

In [5]:
for tool_call in response.tool_calls:
    # Select the tool in tool_dict based on the tool name
    selected_tool = tool_dict[tool_call["name"]] 
    print(tool_call["args"])                   # Arguments for the tool function
    tool_msg = selected_tool.invoke(tool_call) # Result from the tool function
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '인천'}
Asia/Seoul (인천) 현재시각 2026-10-27 15:17:15 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools 사용 가능한 AI 어시스턴트야.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='인천은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 138, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ccaf3fe85e', 'id': 'chatcmpl-EdUmejFXoaRyYZW1ufXY0O2Ok3m6C', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a17c01-6fae-7663-8ed6-72b1d9507b04-0', tool_calls=[{'name': 'get_current_time', 'args': {'time

In [6]:
LLM_with_tools.invoke(messages)

AIMessage(content='현재 인천의 시각은 2026년 10월 27일 15시 17분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 195, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ccaf3fe85e', 'id': 'chatcmpl-EdUmlq0LbKbqNCI1D2dKikgKMowPe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17c01-8e68-74b1-a7fa-687a462ec6ef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 195, 'output_tokens': 26, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [1]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: TSLA, AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [14]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

LLM_with_tools = LLM.bind_tools(tools)

In [15]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = LLM_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 289, 'total_tokens': 360, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_aa07c96156', 'id': 'chatcmpl-Cj0HQOYEtZ9U48imkqbi8ZeKe3bRS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--07e4c439-678a-4602-883a-1dd5e09f090d-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_OhHLVcdud1aKgKyqLcfqYAGo', 'type': 'tool_call'}, {'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1d'}}, 'id': 'call_9EVfnb8GJXYZvSS5nFYQIgWs', 'type': 'tool_call'}] usage_metadata={'input

In [17]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2025-11-04 00:00:00-05:00 | 454.46 | 460.22 | 443.6  |  444.26 | 8.77566e+07 |           0 |              0 |\n| 2025-11-05 00:00:00-05:00 | 452.05 | 466.33 | 440.71 |  462.07 | 8.5573e+07  |           0 |              0 |\n| 2025-11-06 00:00:00-05:00 | 461.96 | 467.45 | 435.09 |  445.91 | 1.09623e+08 |           0 |              0 |\n| 2025-11-07 00:00:00-05:00 | 437.92 | 439.36 | 421.88 |  429.52 | 1.03472e+08 |           0 |              0 |\n| 2025-11-10 00:00:00-05:00 | 439.6  | 449.67 | 433.36 |  445.23 | 7.65159e+07 |           0 |              0 |\n| 2025-11-11 00:00:00-05:00 | 439.4  | 442.49 | 432.36 |  439.62 | 6.05332e+07 |           0 |              0 |\n| 2025-11-12 00:00:00-05:0

In [18]:
LLM_with_tools.invoke(messages)

AIMessage(content='한달 전인 2025년 11월 4일 테슬라(TSLA)의 종가는 $444.26 이었고, 현재 2025년 12월 3일 종가는 $446.74 입니다. \n\n따라서, 테슬라의 주가는 한달 전보다 상승했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 1777, 'total_tokens': 1846, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_aa07c96156', 'id': 'chatcmpl-Cj0RO4JZ4cyCNuLciVkosGZNmuGUi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--3aada0b3-532a-44e5-9763-2827f757773b-0', usage_metadata={'input_tokens': 1777, 'output_tokens': 69, 'total_tokens': 1846, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})